# Retail Sales Exploratory Data Analysis

**Prepared by:** Abinaya  
**Track:** Data Analytics  
**Task:** Level 1, Task 1

## Project objective

Explore transaction, customer, and category data to understand sales patterns and develop practical recommendations. This notebook covers data checks, summary statistics, sales over time, customer demographics, category performance, a correlation heatmap, and an additional weekday analysis.

## Dataset and scope

The dataset is the [Retail Sales Dataset on Kaggle](https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset), file `retail_sales_dataset.csv`. The source describes it as synthetic practice data and lists a CC0 public-domain license. The analysis covers 1,000 transactions dated January 1, 2023 through January 1, 2024. Two transactions fall on January 1, 2024, so the time-trend comparisons below use the 12 complete months and four quarters of 2023.

This is practice data, so recommendations are hypotheses for testing. The file contains product categories but no individual product names, so it supports category rankings rather than a top-10 individual-product list.


## 1. Load the libraries and data

The code looks for the CSV in the Colab session first. If it is not there, Colab will ask you to upload it. Keep `retail_sales_dataset.csv` beside the notebook or upload it when prompted.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

csv_path = Path("/content/retail_sales_dataset.csv")
if not csv_path.exists():
    csv_path = Path("retail_sales_dataset.csv")
if not csv_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = Path(next(iter(uploaded)))
    except ImportError as error:
        raise FileNotFoundError(
            "Place retail_sales_dataset.csv beside this notebook and run this cell again."
        ) from error

df = pd.read_csv(csv_path)
print(f"Loaded {csv_path.name}: {df.shape[0]:,} rows and {df.shape[1]} columns")
display(df.head())


## 2. Inspect data quality

Check the table shape, data types, missing values, duplicates, and whether the recorded transaction total matches quantity multiplied by unit price.


In [ ]:
print("Shape:", df.shape)
print("\nColumn types and non-null counts:")
df.info()

print("\nMissing values:")
display(df.isna().sum().rename("missing values").to_frame())

print("Duplicate rows:", df.duplicated().sum())
amount_matches = df["Total Amount"].eq(df["Quantity"] * df["Price per Unit"])
print(f"Transactions where total equals quantity × unit price: {amount_matches.sum():,}/{len(df):,}")
print("Date range as stored:", df["Date"].min(), "to", df["Date"].max())


### Data-quality finding

The file has 1,000 rows and 9 columns, no missing values, and no duplicate rows. All 1,000 transaction totals equal quantity multiplied by price per unit. Convert `Date` to a datetime type before grouping by month, quarter, or weekday. The two January 1, 2024 records are included in the overall dataset totals but excluded from 2023 time-trend comparisons because they do not make a complete month or quarter.


## 3. Descriptive statistics

Review the mean, median, mode, and standard deviation for numeric columns. `Transaction ID` is an identifier; its statistics describe numbering, not customer or sales behavior.


In [ ]:
numeric = df.select_dtypes(include="number")
stats = pd.DataFrame({
    "mean": numeric.mean(),
    "median": numeric.median(),
    "mode": numeric.mode().iloc[0],
    "standard deviation": numeric.std(),
}).round(2)
display(stats)


The mean transaction amount is **456.00**, while the median is **135.00**. The mean being much higher than the median suggests that some higher-value purchases pull the average upward. Report both values when describing a typical transaction.


## 4. Monthly sales trend

Use only the 12 complete months of 2023. The small number of January 2024 records should not be compared with full months.


In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
sales_2023 = df.loc[df["Date"].dt.year.eq(2023)].copy()
monthly_sales = (
    sales_2023.set_index("Date").resample("MS")["Total Amount"].sum()
)
display(monthly_sales.rename("Revenue").to_frame())

ax = monthly_sales.plot(marker="o", figsize=(10, 5), linewidth=2)
ax.set_title("Monthly Retail Sales in 2023")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
ax.tick_params(axis="x", rotation=30)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Monthly finding

May had the highest recorded monthly revenue (**53,150**). September had the lowest (**23,620**). This single synthetic year suggests months worth investigating, but it is not enough evidence to establish a reliable seasonal pattern.


## 5. Quarterly sales trend

Compare the four complete quarters of 2023. The partial 2024 quarter is intentionally excluded.


In [ ]:
quarterly_sales = (
    sales_2023.set_index("Date").resample("QS")["Total Amount"].sum()
)
quarterly_sales.index = quarterly_sales.index.to_period("Q").astype(str)
display(quarterly_sales.rename("Revenue").to_frame())

ax = quarterly_sales.plot(kind="bar", figsize=(8, 5), color=sns.color_palette("deep", 4))
ax.set_title("Quarterly Retail Sales in 2023")
ax.set_xlabel("Quarter")
ax.set_ylabel("Revenue")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


### Quarterly finding

Quarter 4 had the highest revenue (**126,190**), followed by Quarter 2 (**123,735**). Quarter 3 was lowest (**96,045**). The Q4–Q3 difference is worth checking against a longer history before changing inventory or promotion budgets.


## 6. Customer demographics

The data has a gender and age value on each transaction. Each customer ID appears once, so these results describe this dataset's transactions; they do not measure repeat purchasing or customer lifetime value.


In [ ]:
gender_summary = (
    df.groupby("Gender")
      .agg(transactions=("Transaction ID", "count"),
           revenue=("Total Amount", "sum"),
           average_transaction=("Total Amount", "mean"))
      .sort_values("revenue", ascending=False)
)
display(gender_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
gender_summary["transactions"].plot(kind="bar", ax=axes[0], color=sns.color_palette("deep", 2))
axes[0].set_title("Transactions by Gender")
axes[0].set_xlabel("")
axes[0].set_ylabel("Transactions")
axes[0].tick_params(axis="x", rotation=0)
gender_summary["revenue"].plot(kind="bar", ax=axes[1], color=sns.color_palette("deep", 2))
axes[1].set_title("Revenue by Gender")
axes[1].set_xlabel("")
axes[1].set_ylabel("Revenue")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

age_bins = [17, 24, 34, 44, 54, 64]
age_labels = ["18–24", "25–34", "35–44", "45–54", "55–64"]
df["Age Group"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels)
age_summary = (
    df.groupby("Age Group", observed=False)
      .agg(transactions=("Transaction ID", "count"),
           revenue=("Total Amount", "sum"),
           average_transaction=("Total Amount", "mean"))
)
display(age_summary.round(2))

ax = age_summary["transactions"].plot(kind="bar", figsize=(8, 4), color=sns.color_palette("deep", 5))
ax.set_title("Transactions by Age Group")
ax.set_xlabel("Age group")
ax.set_ylabel("Transactions")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


### Demographic findings

The dataset is close to evenly split: 510 female and 490 male transactions. Revenue is **232,840** for female transactions and **223,160** for male transactions. Average transaction amounts are nearly the same (**456.55** and **455.43**). The 45–54 age group has the most transactions (225), while the 18–24 group has the highest average transaction amount (501.01). Treat these as descriptive patterns in synthetic data, not evidence that gender or age caused spending differences.


## 7. Product category performance

This file has a `Product Category` column but no individual product name or SKU. The charts therefore compare the three available categories. A separate dataset with product-level fields is needed for a valid top-10 product ranking.


In [ ]:
category_summary = (
    df.groupby("Product Category")
      .agg(transactions=("Transaction ID", "count"),
           units_sold=("Quantity", "sum"),
           revenue=("Total Amount", "sum"),
           average_transaction=("Total Amount", "mean"))
      .sort_values("revenue", ascending=False)
)
category_summary["revenue_share_pct"] = 100 * category_summary["revenue"] / category_summary["revenue"].sum()
display(category_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
category_summary.sort_values("revenue")["revenue"].plot(kind="barh", ax=axes[0], color=sns.color_palette("deep", 3))
axes[0].set_title("Revenue by Category")
axes[0].set_xlabel("Revenue")
axes[0].set_ylabel("")
category_summary.sort_values("units_sold")["units_sold"].plot(kind="barh", ax=axes[1], color=sns.color_palette("deep", 3))
axes[1].set_title("Units Sold by Category")
axes[1].set_xlabel("Units")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()


### Category finding

Clothing led unit volume with **894 units** (35.6% of all units). Electronics produced the most revenue (**156,905**, or 34.4% of revenue), slightly above Clothing (**155,580**). The two categories are close in revenue; check product costs and margins before prioritizing either one.


## 8. Correlation heatmap

Use numeric behavior and sales fields. Exclude `Transaction ID` because its sequence does not represent a meaningful quantity.


In [ ]:
correlation_columns = ["Age", "Quantity", "Price per Unit", "Total Amount"]
corr = df[correlation_columns].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="vlag", center=0, vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Between Numeric Sales Fields")
plt.tight_layout()
plt.show()


### Correlation finding

`Price per Unit` and `Total Amount` have a strong positive correlation (**0.85**); `Quantity` and `Total Amount` have a moderate positive correlation (**0.37**). This is expected because `Total Amount` is calculated as quantity multiplied by unit price. Age has little linear correlation with transaction amount (**−0.06**) in this dataset. Correlation describes association and does not establish cause.


## 9. Additional analysis: average transaction amount by weekday

This view checks whether transaction value differs across days of the week in 2023.


In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sales_2023["Weekday"] = sales_2023["Date"].dt.day_name()
weekday_summary = (
    sales_2023.groupby("Weekday")
      .agg(transactions=("Transaction ID", "count"),
           revenue=("Total Amount", "sum"),
           average_transaction=("Total Amount", "mean"))
      .reindex(weekday_order)
)
display(weekday_summary.round(2))

ax = weekday_summary["average_transaction"].plot(kind="bar", figsize=(9, 4), color=sns.color_palette("deep", 7))
ax.set_title("Average Transaction Amount by Weekday in 2023")
ax.set_xlabel("Weekday")
ax.set_ylabel("Average transaction amount")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


### Weekday finding

Saturday had the highest average transaction amount (**525.43**); Wednesday had the lowest (**422.81**). This is a useful hypothesis for a small promotion or staffing test, but the synthetic data and one-year window do not establish a recurring weekly pattern.


## 10. Recommendations

1. **Plan a small category-level inventory and promotion test.** Clothing has the highest unit volume, while Electronics has the highest revenue. Compare gross margins, returns, and stockouts before committing more budget to either category.
2. **Investigate the 2023 monthly and quarterly pattern.** May was the strongest month and September the weakest; Q4 was the strongest quarter and Q3 the weakest. Compare at least two years of real transaction data and account for promotions or stock availability before treating these as seasonal effects.
3. **Test a weekend offer before changing staffing.** Saturday's average transaction value was highest in this sample. Run a limited, measured test and compare conversion, order value, and margin against other days.
4. **Avoid demographic targeting based on this file alone.** Gender average spend is almost identical, and the age pattern is descriptive. Validate any customer segment with real, repeat-purchase data and appropriate privacy practices.

## Conclusion

This dataset is complete and internally consistent for a beginner EDA project. It supports analysis of sales timing, customer demographics, and category performance. Its synthetic nature, one-year coverage, single transaction per customer ID, and lack of product-level names limit business conclusions. The next improvement would be a multi-year dataset with product IDs, costs, promotions, and repeat customer purchases.
